# Passo 7 – Generalização: mensagens de WhatsApp (2018) e afirmações checadas

Este notebook **acrescenta** duas bases ao estudo. Nada dos passos anteriores é refeito.

| Base | O que é | Uso aqui |
|---|---|---|
| **FakeWhatsApp.Br** (Cabral et al., ICEIS 2021) | Mensagens de grupos públicos de política no WhatsApp, eleições de **2018**. Rótulo: desinformação (1) × não desinformação (0) | Teste externo com as duas classes; treino no formato de mensagem |
| **X-FACT, parte brasileira** (Gupta; Srikumar, ACL 2021) | Afirmações checadas por Lupa e Aos Fatos (2016–2020) | Controle **C3**: afirmações verdadeiras curtas; e teste verdadeiro × falso |

**Experimentos:**

1. **E1:** modelos treinados no **Fake.Br** (parâmetros do Passo 4) testados no FakeWhatsApp.Br.
2. **E2:** NB, SVM e XGBoost treinados no **FakeWhatsApp.Br**, com a mesma busca de parâmetros, validação cruzada em grupos e McNemar.
3. **E3:** treino **misto** (Fake.Br + FakeWhatsApp.Br).
4. **E4:** todos os modelos avaliados nos **boatos de 2022** (FactPolCheckBr), nas **falas autênticas (C2)** e no **X-FACT (C3)**.

**Cuidados declarados:**
- O FakeWhatsApp.Br teve **um só anotador** (sem kappa).
- A classe 0 é **"não desinformação"**: inclui opinião e humor. Não é o mesmo que "verdadeiro".
- Mensagens virais têm **variações quase iguais**. Elas são agrupadas (similaridade ≥ 0,8) e o grupo inteiro fica do mesmo lado da divisão.

**Tempo:** cerca de 15 a 30 minutos no Colab.

**Pré-requisito:** notebooks 00 a 04 e 02b executados.

In [ ]:
import os, re, csv, glob, json, ast, time, unicodedata, subprocess, itertools, warnings
import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_TCC = '/content/drive/MyDrive/TCC'
    BRUTOS = '/content/dados_brutos'
except ImportError:
    PASTA_TCC = os.path.abspath('TCC')
    BRUTOS = os.path.join(PASTA_TCC, 'dados', 'brutos')
PROC = os.path.join(PASTA_TCC, 'dados', 'processados')
RES = os.path.join(PASTA_TCC, 'resultados')
REV = os.path.join(RES, 'revisao')
SEED = 42
np.random.seed(SEED)

REPOS = {'FakeWhatsApp.Br': 'https://github.com/cabrau/FakeWhatsApp.Br',
         'x-fact': 'https://github.com/utahnlp/x-fact'}
os.makedirs(BRUTOS, exist_ok=True)
versoes = {}
for nome, url in REPOS.items():
    destino = os.path.join(BRUTOS, nome)
    if not os.path.isdir(destino):
        subprocess.run(['git', 'clone', '--depth', '1', '-q', url, destino], check=True)
    versoes[nome] = subprocess.run(['git', '-C', destino, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Versões (commit):', versoes)
with open(os.path.join(RES, 'versoes_bases_passo7.txt'), 'w') as f:
    for n, v in versoes.items():
        f.write(f'{n}: {REPOS[n]} (commit {v})\n')

In [ ]:
import nltk
# O Colab apaga estes recursos quando a sessão acaba, então baixamos de novo (rápido)
for recurso in ['stopwords', 'rslp']:
    nltk.download(recurso, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

NEGACOES = {'não', 'nem', 'nunca', 'jamais', 'nenhum', 'nenhuma', 'sem', 'nada', 'ninguém'}
STOP = set(stopwords.words('portuguese')) - NEGACOES
STEM = RSLPStemmer()

RE_URL = re.compile(r'(https?://\S+|www\.\S+)', re.I)
RE_MENCAO = re.compile(r'@\w+')
RE_HASHTAG = re.compile(r'#\w+')
RE_MARCA = re.compile(r'\(sic\.?\)|\[sic\.?\]|\(\.\.\.\)|\[\.\.\.\]', re.I)
RE_NUM = re.compile(r'\d+([.,]\d+)*')
RE_NAOLETRA = re.compile(r'[^a-zA-ZÀ-ÿ ]')

def normalizar(texto):
    t = str(texto)
    t = unicodedata.normalize('NFC', t)
    t = RE_URL.sub(' ', t)
    t = RE_MARCA.sub(' ', t)
    t = RE_MENCAO.sub(' ', t)
    t = RE_HASHTAG.sub(' ', t)
    t = t.lower()
    t = RE_NUM.sub(' num ', t)
    t = RE_NAOLETRA.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def tokenizar(texto):
    return [p for p in normalizar(texto).split() if len(p) > 1 or p == 'e']

def sem_stopwords(tokens):
    return [t for t in tokens if t not in STOP]

def preparar(texto):
    toks = sem_stopwords(tokenizar(texto))
    return ' '.join(toks), ' '.join(STEM.stem(t) for t in toks)

def preparar_proc(texto):
    return ' '.join(sem_stopwords(tokenizar(texto)))

## 1. FakeWhatsApp.Br: limpeza e agrupamento de mensagens quase iguais

Regras:
- Só mensagens **rotuladas** (o rótulo -1 significa "não rotulada").
- Texto repetido **exatamente igual** entra uma vez só.
- Pelo menos **5 palavras** (mesmo critério dos autores para mensagens virais).
- Mensagens com similaridade de cosseno **≥ 0,8** (TF-IDF) formam um **grupo**. O grupo fica inteiro no treino ou no teste.

In [ ]:
fwa = pd.read_csv(os.path.join(BRUTOS, 'FakeWhatsApp.Br', 'data', '2018', 'fakeWhatsApp.BR_2018.csv'), low_memory=False)
n0 = len(fwa)
fwa = fwa[fwa['misinformation'] != -1]
n1 = len(fwa)
fwa = fwa.drop_duplicates('text')
n2 = len(fwa)
fwa = fwa[fwa['words'] >= 5].reset_index(drop=True)
fwa['rotulo'] = fwa['misinformation'].astype(int)
fwa['texto'] = fwa['text'].astype(str)
fwa['texto_proc'] = [preparar_proc(t) for t in fwa['texto']]
fwa = fwa[fwa['texto_proc'].str.len() > 0].reset_index(drop=True)
fwa['n_palavras_orig'] = fwa['texto'].str.split().str.len()
print(f'Mensagens: {n0} no total | {n1} rotuladas | {n2} sem repetição exata | {len(fwa)} finais')
print('Rótulos:', fwa['rotulo'].value_counts().to_dict(), '| mediana de palavras:', fwa['n_palavras_orig'].median())

In [ ]:
# Agrupa quase-duplicatas (cosseno >= 0,8), em blocos para caber na memória
X = TfidfVectorizer(min_df=1, sublinear_tf=True).fit_transform(fwa['texto_proc'])
linhas, colunas = [], []
for ini in range(0, X.shape[0], 1000):
    sim = (X[ini:ini + 1000] @ X.T).tocoo()
    m = sim.data >= 0.8
    linhas.append(sim.row[m] + ini); colunas.append(sim.col[m])
r, c = np.concatenate(linhas), np.concatenate(colunas)
grafo = coo_matrix((np.ones(len(r)), (r, c)), shape=(X.shape[0], X.shape[0]))
n_grupos, fwa['grupo'] = connected_components(grafo, directed=False)
tam = fwa['grupo'].value_counts()
print(f'{n_grupos} grupos para {len(fwa)} mensagens | maior grupo: {tam.max()} | mensagens em grupos com 2 ou mais: {int(tam[tam > 1].sum())}')
mistos = fwa.groupby('grupo')['rotulo'].nunique()
print('Grupos com rótulos diferentes dentro (sinal de inconsistência na anotação):', int((mistos > 1).sum()))

In [ ]:
# Divisão 80/20 por grupo
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
i_tr, i_te = next(gss.split(fwa, fwa['rotulo'], groups=fwa['grupo']))
fwa_tr, fwa_te = fwa.iloc[i_tr].reset_index(drop=True), fwa.iloc[i_te].reset_index(drop=True)
assert set(fwa_tr.grupo).isdisjoint(set(fwa_te.grupo))
print('FakeWhatsApp.Br treino:', fwa_tr['rotulo'].value_counts().to_dict(), '| teste:', fwa_te['rotulo'].value_counts().to_dict())

## 2. X-FACT (parte brasileira) e controle C2

- **X-FACT:** só Lupa (piaui.folha.uol.com.br) e Aos Fatos. Ficam os rótulos **true** (verdadeiro) e **false** (falso). Os intermediários saem, para não haver ambiguidade.
- **C2:** falas e textos autênticos descartados na revisão dos boatos (mesmo critério do Passo 6).

In [ ]:
partes = []
for arq in ['train.all.tsv', 'dev.all.tsv', 'test.all.tsv', 'ood.tsv']:
    d = pd.read_csv(os.path.join(BRUTOS, 'x-fact', 'data', 'x-fact', arq), sep='\t', quoting=csv.QUOTE_NONE, on_bad_lines='skip')
    partes.append(d[(d.language == 'pt') & d.site.isin(['piaui.folha.uol.com.br', 'aosfatos.org'])])
xf = pd.concat(partes, ignore_index=True)
xf = xf[xf.label.isin(['true', 'false'])].drop_duplicates('claim').reset_index(drop=True)
xf['rotulo'] = (xf.label == 'false').astype(int)
xf['texto'] = xf['claim'].astype(str)
xf['texto_proc'] = [preparar_proc(t) for t in xf['texto']]
xf = xf[xf['texto_proc'].str.len() > 0].reset_index(drop=True)
print('X-FACT BR:', xf.label.value_counts().to_dict(), '| mediana de palavras:', xf.texto.str.split().str.len().median())

decisoes = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(REV, 'revisao_claude_lote_*.csv')))])
cand = pd.read_csv(os.path.join(REV, 'revisao_completa.csv'))[['id', 'trecho']]
AUTENTICO = ['fala ou texto verdadeiro', 'conteúdo original reaproveitado', 'fala real', 'tuíte original', 'fala verdadeira']
mask = (decisoes.decisao == 'descartar') & decisoes.motivo.fillna('').apply(lambda m: any(a in m for a in AUTENTICO))
c2 = decisoes[mask][['id']].merge(cand, on='id').rename(columns={'trecho': 'texto'})
c2['texto_proc'] = [preparar_proc(t) for t in c2['texto']]
print('C2 (falas autênticas):', len(c2))

fake = pd.read_csv(os.path.join(PROC, 'fakebr_prep.csv'))
fpc = pd.read_csv(os.path.join(PROC, 'fpc_prep.csv'))
for d in (fake, fpc):
    d['texto_proc'] = d['texto_proc'].fillna('')
fake['rotulo'] = fake['rotulo'].astype(int)
fk_tr = fake[fake.particao == 'treino'].reset_index(drop=True)
fk_te = fake[fake.particao == 'teste'].reset_index(drop=True)
print('Fake.Br treino/teste:', len(fk_tr), len(fk_te), '| boatos 2022:', len(fpc))

## 3. Modelos

Mesmas configurações do Passo 4 (TF-IDF + NB / SVM linear / XGBoost), sem stemming (a versão escolhida no Passo 4).

In [ ]:
def pipe(nome):
    if nome == 'Naive Bayes':
        return Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True)), ('clf', MultinomialNB())]), \
               {'tfidf__ngram_range': [(1, 1), (1, 2)], 'clf__alpha': [0.1, 0.5, 1.0]}
    if nome == 'SVM':
        return Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=50000, sublinear_tf=True)), ('clf', LinearSVC(random_state=SEED, max_iter=5000))]), \
               {'tfidf__ngram_range': [(1, 1), (1, 2)], 'clf__C': [0.1, 1, 10]}
    return Pipeline([('tfidf', TfidfVectorizer(min_df=2, max_features=10000, sublinear_tf=True)),
                     ('clf', XGBClassifier(tree_method='hist', n_jobs=1, random_state=SEED, eval_metric='logloss', learning_rate=0.1))]), \
           {'clf__n_estimators': [200, 400], 'clf__max_depth': [4, 6]}

MODELOS = ['Naive Bayes', 'SVM', 'XGBoost']
with open(os.path.join(RES, 'melhores_parametros.json')) as f:
    PARAMS_FK = {k.split('|')[0]: {p: ast.literal_eval(v) for p, v in d.items()} for k, d in json.load(f).items() if k.endswith('sem_stem')}

def taxas(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {'f1_falso': round(f1_score(y, p, zero_division=0), 4), 'recall': round(tp / (tp + fn), 4),
            'taxa_fp': round(fp / (fp + tn), 4)}

def prop(p):
    k, n = int(np.sum(p)), len(p)
    lo, hi = proportion_confint(k, n, method='wilson')
    return round(k / n, 3), f'{lo:.3f}–{hi:.3f}'

## E2 – Busca de parâmetros no FakeWhatsApp.Br (validação cruzada em grupos)

In [ ]:
buscas_fwa, linhas = {}, []
for nome in MODELOS:
    t0 = time.time()
    p, grade = pipe(nome)
    gs = GridSearchCV(p, grade, cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED), scoring='f1', n_jobs=-1)
    gs.fit(fwa_tr['texto_proc'], fwa_tr['rotulo'], groups=fwa_tr['grupo'])
    buscas_fwa[nome] = gs
    linhas.append({'modelo': nome, 'f1_cv_media': round(gs.best_score_, 4),
                   'f1_cv_desvio': round(gs.cv_results_['std_test_score'][gs.best_index_], 4),
                   'melhores_parametros': str(gs.best_params_), 'minutos': round((time.time() - t0) / 60, 1)})
    print(linhas[-1])
quadro_cv_fwa = pd.DataFrame(linhas)
quadro_cv_fwa

## Treino dos modelos finais (três origens de treino)

- **Fake.Br:** parâmetros do Passo 4.
- **FakeWhatsApp.Br:** parâmetros da busca acima.
- **Misto:** Fake.Br (treino) + FakeWhatsApp.Br (treino), com os parâmetros do FakeWhatsApp.Br.

In [ ]:
modelos = {}
for nome in MODELOS:
    p, _ = pipe(nome)
    modelos[('Fake.Br', nome)] = clone(p).set_params(**PARAMS_FK[nome]).fit(fk_tr['texto_proc'], fk_tr['rotulo'])
    modelos[('FakeWhatsApp.Br', nome)] = buscas_fwa[nome].best_estimator_
    misto_x = pd.concat([fk_tr['texto_proc'], fwa_tr['texto_proc']])
    misto_y = pd.concat([fk_tr['rotulo'], fwa_tr['rotulo']])
    modelos[('Misto', nome)] = clone(p).set_params(**buscas_fwa[nome].best_params_).fit(misto_x, misto_y)
print('Modelos treinados:', len(modelos))

## E1, E3 e E4 – Quadro geral

Colunas:
- **Fake.Br teste** e **FakeWhatsApp.Br teste:** F1 da classe falsa e taxa de falsos positivos.
- **X-FACT:** F1 e falsos positivos em afirmações verdadeiras × falsas.
- **Boatos 2022:** recall (só há falsos).
- **C2:** falsos positivos nas falas autênticas.
- **Separação 2022** = recall nos boatos − falsos positivos no C2. Zero = não separa.

In [ ]:
linhas = []
for (origem, nome), m in modelos.items():
    r_fk = taxas(fk_te['rotulo'], m.predict(fk_te['texto_proc']))
    r_fw = taxas(fwa_te['rotulo'], m.predict(fwa_te['texto_proc']))
    r_xf = taxas(xf['rotulo'], m.predict(xf['texto_proc']))
    rec_b, ic_b = prop(m.predict(fpc['texto_proc']))
    fp_c2, ic_c2 = prop(m.predict(c2['texto_proc']))
    linhas.append({'treino': origem, 'modelo': nome,
                   'f1_fakebr': r_fk['f1_falso'], 'fp_fakebr': r_fk['taxa_fp'],
                   'f1_fakewhatsapp': r_fw['f1_falso'], 'fp_fakewhatsapp': r_fw['taxa_fp'],
                   'f1_xfact': r_xf['f1_falso'], 'fp_xfact': r_xf['taxa_fp'],
                   'recall_boatos_2022': rec_b, 'ic_boatos': ic_b, 'fp_c2': fp_c2, 'ic_c2': ic_c2,
                   'separacao_2022': round(rec_b - fp_c2, 3)})
quadro_geral = pd.DataFrame(linhas)
quadro_geral

## McNemar no teste do FakeWhatsApp.Br (modelos treinados nele)

Mesmo procedimento do Passo 4: 3 comparações, nível 0,05/3 ≈ 0,0167 (Bonferroni).

In [ ]:
pred = {n: modelos[('FakeWhatsApp.Br', n)].predict(fwa_te['texto_proc']) for n in MODELOS}
y = fwa_te['rotulo'].values
linhas = []
for a, b in itertools.combinations(MODELOS, 2):
    oa, ob = pred[a] == y, pred[b] == y
    res = mcnemar([[int((oa & ob).sum()), int((oa & ~ob).sum())], [int((~oa & ob).sum()), int((~oa & ~ob).sum())]], exact=True)
    linhas.append({'A': a, 'B': b, 'so_A_acerta': int((oa & ~ob).sum()), 'so_B_acerta': int((~oa & ob).sum()),
                   'p_valor': res.pvalue, 'significativo_bonferroni': res.pvalue < 0.05 / 3})
quadro_mc_fwa = pd.DataFrame(linhas)
quadro_mc_fwa

## Palavras mais pesadas do SVM treinado no FakeWhatsApp.Br

Para checar atalhos (por exemplo, palavras de formatação de mensagem, nomes de grupos ou pedidos de "compartilhe").

In [ ]:
est = modelos[('FakeWhatsApp.Br', 'SVM')]
nomes = np.array(est.named_steps['tfidf'].get_feature_names_out())
coef = est.named_steps['clf'].coef_[0]
ordem = np.argsort(coef)
top_fwa = pd.DataFrame({'palavras_desinformacao': nomes[ordem[::-1][:25]], 'palavras_nao_desinformacao': nomes[ordem[:25]]})
top_fwa

## Salvando

In [ ]:
quadro_cv_fwa.to_csv(os.path.join(RES, 'quadro11_validacao_cruzada_fakewhatsapp.csv'), index=False)
quadro_geral.to_csv(os.path.join(RES, 'quadro12_generalizacao.csv'), index=False)
quadro_mc_fwa.to_csv(os.path.join(RES, 'quadro13_mcnemar_fakewhatsapp.csv'), index=False)
top_fwa.to_csv(os.path.join(RES, 'top_palavras_SVM_fakewhatsapp.csv'), index=False)
pd.DataFrame([{'base': 'FakeWhatsApp.Br', 'mensagens': len(fwa), 'grupos': int(fwa.grupo.nunique()),
               'desinformacao': int(fwa.rotulo.sum()), 'nao_desinformacao': int((fwa.rotulo == 0).sum()),
               'mediana_palavras': fwa.n_palavras_orig.median()},
              {'base': 'X-FACT BR', 'mensagens': len(xf), 'grupos': np.nan, 'desinformacao': int(xf.rotulo.sum()),
               'nao_desinformacao': int((xf.rotulo == 0).sum()), 'mediana_palavras': xf.texto.str.split().str.len().median()}
             ]).to_csv(os.path.join(RES, 'quadro14_bases_passo7.csv'), index=False)
print('Resultados salvos em', RES)